# Temporal v1 Fixed Evaluation

Compare trained temporal cache-probe checkpoints on the same fixed validation set. This notebook is aligned with the current tick-regression target: each future chunk predicts normalized low/high absolute ticks plus up/down/path class logits. Confusion matrices come from the classification heads rather than decoded predicted header bits.

Default selection is the two linear-probe runs you trained: `epoch1` and `latest`. Update `RUN_NAME_MATCHES` or set `CHECKPOINTS` explicitly if needed.


In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path.cwd()
while REPO_ROOT.name != 'quant-research-workbench' and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from research.temporal_event_model.v1.evaluate_cache_probe_checkpoints import evaluate_checkpoints

CACHE_ROOT = Path(r'D:\market-data\prepared\event_sample_cache\cache_v2_cycle_20260619_134422')
CHECKPOINT_ROOT = Path(r'D:\TradingML\runtimes\temporal_event_model\v1\cache_price_probe_laptop')
OUTPUT_DIR = Path(r'D:\TradingML\runtimes\temporal_event_model\v1\cache_price_probe_laptop_eval\extrema-fixed-shard1-10x1024')

# Leave CHECKPOINTS empty to select checkpoint_latest.pt files by run-name match.
CHECKPOINTS = []
RUN_NAME_MATCHES = [
    'v1-extrema-probe-v20-epoch1',
    'v1-extrema-probe-v20-latest',
]

BATCH_SIZE = 1024
VALIDATION_BATCHES = 10
SEED = 20260621
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_DTYPE = torch.bfloat16 if DEVICE.type == 'cuda' else None

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'device={DEVICE}')
print(f'cache_root={CACHE_ROOT}')
print(f'checkpoint_root={CHECKPOINT_ROOT}')
print(f'output_dir={OUTPUT_DIR}')


In [ ]:
def select_checkpoints(checkpoint_root: Path, run_name_matches: list[str], explicit: list[str | Path]) -> list[Path]:
    if explicit:
        return [Path(path) for path in explicit]
    candidates = sorted(
        checkpoint_root.glob('*/checkpoints/checkpoint_latest.pt'),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    selected = []
    seen = set()
    for pattern in run_name_matches:
        for path in candidates:
            run_name = path.parent.parent.name
            if pattern in run_name and path not in seen:
                selected.append(path)
                seen.add(path)
                break
    if not selected:
        selected = candidates[:3]
    return selected

checkpoint_paths = select_checkpoints(CHECKPOINT_ROOT, RUN_NAME_MATCHES, CHECKPOINTS)
print('Checkpoints:')
for path in checkpoint_paths:
    print(' ', path)

result = evaluate_checkpoints(
    checkpoint_paths=checkpoint_paths,
    cache_root=CACHE_ROOT,
    validation_split='train',
    validation_start_shard=1,
    validation_max_shards=1,
    validation_batches=VALIDATION_BATCHES,
    batch_size=BATCH_SIZE,
    seed=SEED,
    device=DEVICE,
    amp_dtype=AMP_DTYPE,
    output_dir=OUTPUT_DIR,
)
result_path = Path(result['result_path'])
print(f'wrote {result_path}')


In [ ]:
# To skip recomputation in later notebook runs, comment the previous cell and load the saved result here.
result_path = OUTPUT_DIR / 'fixed_eval_results.json'
result = json.loads(result_path.read_text(encoding='utf-8'))
up_class_names = result['up_class_names']
down_class_names = result['down_class_names']
path_class_names = result['path_class_names']
summary = result['summary']
pd.DataFrame(summary)


In [ ]:
def short_name(run: dict) -> str:
    name = run['summary']['run_name']
    replacements = [
        ('v1-extrema-probe-v20-', ''),
        ('-1train1val-5ep-bs512-laptop', ''),
        ('-1train1val-5ep-bs1024-laptop', ''),
        ('v1-cache-probe-v20-', ''),
    ]
    for old, new in replacements:
        name = name.replace(old, new)
    return name

rows = []
for run in result['runs']:
    row = {'run': short_name(run), **run['summary']}
    rows.append(row)
summary_df = pd.DataFrame(rows)
summary_df


In [ ]:
metric_columns = [
    ('loss', 'Total loss'),
    ('regression_mse', 'Tick regression MSE'),
    ('classification_loss', 'Class CE loss'),
    ('path_accuracy_pct', 'Path accuracy %'),
    ('low_tick_mae', 'Low MAE ticks'),
    ('high_tick_mae', 'High MAE ticks'),
    ('low_price_mae_dollars', 'Low MAE $'),
    ('high_price_mae_dollars', 'High MAE $'),
]

fig, axes = plt.subplots(1, len(metric_columns), figsize=(4.0 * len(metric_columns), 4.5))
if len(metric_columns) == 1:
    axes = [axes]
labels = summary_df['run'].tolist()
x = np.arange(len(labels))
for ax, (column, title) in zip(axes, metric_columns):
    if column not in summary_df.columns:
        ax.set_visible(False)
        continue
    values = summary_df[column].astype(float).to_numpy()
    ax.bar(x, values, color='#4C78A8')
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=25, ha='right')
    ax.grid(axis='y', alpha=0.25)
    for idx, value in enumerate(values):
        ax.text(idx, value, f'{value:.3g}', ha='center', va='bottom', fontsize=8)
fig.suptitle('Fixed validation summary', y=1.02)
fig.tight_layout()
plt.show()


In [ ]:
def normalize_rows(matrix: np.ndarray) -> np.ndarray:
    matrix = np.asarray(matrix, dtype=np.float64)
    denom = matrix.sum(axis=1, keepdims=True)
    return np.divide(matrix, np.maximum(denom, 1.0)) * 100.0


def plot_confusion(ax, matrix, labels, title, normalize=True):
    matrix = np.asarray(matrix, dtype=np.float64)
    values = normalize_rows(matrix) if normalize else matrix
    im = ax.imshow(values, cmap='Blues', vmin=0)
    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=35, ha='right')
    ax.set_yticklabels(labels)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(title)
    threshold = values.max() * 0.55 if values.size else 0
    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            if normalize:
                text = f'{values[i, j]:.1f}%\n({int(matrix[i, j]):,})'
            else:
                text = f'{int(values[i, j]):,}'
            ax.text(j, i, text, ha='center', va='center', color='white' if values[i, j] > threshold else 'black', fontsize=8)
    return im


In [ ]:
matrix_specs = [
    ('upside', up_class_names, 'Upside'),
    ('downside', down_class_names, 'Downside'),
    ('path', path_class_names, 'Path'),
]

for matrix_key, class_labels, title in matrix_specs:
    fig, axes = plt.subplots(1, len(result['runs']), figsize=(5.0 * len(result['runs']), 4.5), squeeze=False)
    for col, run in enumerate(result['runs']):
        matrix = run['confusion_matrices'][matrix_key]
        plot_confusion(axes[0, col], matrix, class_labels, f'{short_name(run)}\n{title}', normalize=True)
    fig.suptitle(f'{title} confusion matrices - row normalized with counts', y=1.04)
    fig.tight_layout()
    plt.show()


In [ ]:
# Raw metrics for deeper inspection.
for run in result['runs']:
    print('\n' + '=' * 100)
    print(short_name(run))
    print(json.dumps(run['metrics'], indent=2))
